# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdityaPrakash-Kaizu07/Flyrank-AI-intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AdityaPrakash-Kaizu07/Flyrank-AI-intern"
REPO_DIR = "Flyrank-AI-intern"

if IN_COLAB:
    # In Colab, clone the repo if it doesn't exist yet
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

    # Install dependencies if you have a requirements.txt
    if os.path.exists("requirements.txt"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # On a local machine, find the repo root from wherever this notebook started
    # (Checking for 'README.md' or another unique folder/file at your repo root)
    while not os.path.isfile("README.md") and os.getcwd() != "/":
        os.chdir("..")
print("Working dir:", os.getcwd())
# Ensure we are actually at the repo root by asserting a known file/folder exists
assert os.path.exists("README.md"), "Root file not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/Flyrank-AI-intern
Starter data found. You're ready.


In [2]:
import pandas as pd

# Use a relative path since your working directory is already at the repo root
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print(df.shape)
print(df.columns.tolist())
print(df.head())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.0

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My task is **multi-class classification**. I will classify each content
page into one of three priority tiers: **High**, **Medium**, or **Low**
opportunity for refresh.

The model learns from signals like declining CTR, content recency, and
search volume to predict which pages an editorial team should prioritize
for update. This is a 3-class problem because editors make discrete
decisions, not continuous rankings.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target: `priority_tier` (High/Medium/Low).

I will **synthesize** this label from signals already in the CSV using
a defensible editorial rule:

- **High:** declining (trend_pct < -30) AND old (days_since_last_update > 90)
  AND/OR high search volume (top 25%)
- **Medium:** meets at least one of the above criteria
- **Low:** none of the above

This is NOT observed ground truth (we don't have edit history or outcome
data). Instead, it represents editorial judgment: which pages *should* be
refreshed based on opportunity signals. The model learns to predict this
prioritization—and potentially refine or generalize it.?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: **Precision @ K** and **F1-score per class**.

Specifically:
- Do the top-ranked (High-priority) pages actually show the strongest
  combination of decline + age + search volume?
- Can the model generalize better than the rigid rule?
- Does it flag edge cases the rule misses?

We will compare the model's predictions against the baseline rule:
if the model sorts pages differently but makes editorial sense, that's
a win.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Debug: check actual ranges
print("Data ranges:")
print(f"  days_since_last_update: {df['days_since_last_update'].min()} → {df['days_since_last_update'].max()}")
print(f"  trend_pct: {df['trend_pct'].min():.1f} → {df['trend_pct'].max():.1f}")
print(f"  search_volume: {df['search_volume'].min():.1f} → {df['search_volume'].max():.1f}")

# Adjusted thresholds (realistic for this data)
def assign_priority(row):
    declining = row['trend_pct'] < -30  # More negative = worse decline
    old = row['days_since_last_update'] > 90  # Over 3 months old (not 1 year)
    high_volume = row['search_volume'] > df['search_volume'].quantile(0.75)  # Top 25% of search volume

    # High: meets 2+ criteria strongly
    if sum([declining, old, high_volume]) >= 2:
        return 'High'
    # Medium: meets at least 1 criterion
    elif sum([declining, old, high_volume]) >= 1:
        return 'Medium'
    else:
        return 'Low'

df['priority_tier'] = df.apply(assign_priority, axis=1)

print("\nUnit of Analysis: One row = one content page")
print("\nSample rows:")
display(df[['content_id', 'search_volume', 'days_since_last_update', 'trend_pct',
            'ctr', 'avg_position', 'priority_tier']].head(10))

print(f"\nLabel distribution:")
print(df['priority_tier'].value_counts())


Data ranges:
  days_since_last_update: 1 → 373
  trend_pct: -100.0 → 44900.0
  search_volume: 0.0 → 74000.0

Unit of Analysis: One row = one content page

Sample rows:


,content_id,search_volume,days_since_last_update,trend_pct,ctr,avg_position,priority_tier
0,content_304f48230142,10.0,20,-41.4,0.76,10.6,Medium
1,content_a1fb4e703a9e,90.0,25,-57.7,0.05,20.3,High
2,content_9aa793d4d895,0.0,20,-60.9,0.09,36.5,Medium
3,content_331d6c4de07b,10.0,22,-13.8,0.49,6.2,Low
4,content_d99b7a2d90ca,0.0,14,-34.7,0.13,44.0,Medium
5,content_d4084a4bc775,720.0,20,-38.9,0.03,8.5,High
6,content_9a34b442b552,0.0,20,-92.3,0.00,7.0,Medium
7,content_a63219c6e95a,590.0,22,0.6,0.06,21.2,Medium
8,content_5e6c160719bc,0.0,20,-58.8,0.09,46.0,Medium
9,content_c27558df2b0c,0.0,104,-29.2,0.16,4.9,Medium



Label distribution:
priority_tier
Medium    13554
Low        8531
High       7915
Name: count, dtype: int64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (if declining AND old AND high_volume → High) is rigid
and binary. Real editorial triage requires nuance:

1. **Non-linear interactions:** A page that's very old but stable might
   still need refresh for brand consistency (not captured by AND logic)
2. **Trade-offs:** High volume + slight decline might outrank low volume
   + steep decline (depends on context)
3. **Generalization:** The rule is locked in; a model learns patterns
   and adapts to new data
4. **Confidence:** A classifier gives probabilities, not just buckets—
   editors can see "this page is 89% likely High-priority"

Without ML, we hardcode every exception. With ML, we learn the pattern
from data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

✓ Task type named: Multi-class classification

✓ Target defined: priority_tier (High/Medium/Low) — synthesized from signals

✓ Success metric named: Precision @ K, F1-score, comparison vs. baseline rule

✓ Dataframe shown: 10 real rows with label distribution (High: 7,915, Medium: 13,554, Low: 8,531)

✓ Why ML beats rules explained: Non-linear interactions, trade-offs, generalization, confidence

✓ Honest language: Synthetic label (editorial judgment), not ground truth